# RNA–protein dynamics in HSPCs

This analysis uses paired RNA and protein measurements from hematopoietic stem and progenitor cells. We examine reconstructed dynamics, perturbation of the megakaryocyte progenitor (MkP) program and the ITGA2B–CD41 response (Figure 4).


In [ ]:
from pathlib import Path
import json

from mofi_notebook import setup_notebook, suppress_live_plot_output

ROOT, DEVICE, _ = setup_notebook()
print(f"Device: {DEVICE}")


## Continuous-time RNA–protein dynamics


In [ ]:
from CytoBridge.reproducibility import require_paper_data, run_real_downstream

DATASET = "hspc_31800"
paths = require_paper_data(DATASET, ROOT)
paper_output = ROOT / "results" / "figure_4_paper"

with suppress_live_plot_output():
    downstream = run_real_downstream(
        DATASET,
        output_dir=paper_output,
        selected_figures=[],
        device=DEVICE,
        sample_max_cells=None,
        reuse_cache=True,
        repo_root=ROOT,
    )
run_dir = Path(downstream["run_dir"])
run_dir


## Observed and interpolated cell states

Time slices show the measured populations alongside intermediate states predicted by MOFI.


In [ ]:
from CytoBridge.manuscript import render_hspc_time_slices
from CytoBridge.reproducibility import write_resolved_analysis_config

local_cfg = write_resolved_analysis_config(
    DATASET,
    output_dir=paper_output,
    config_path=ROOT / "results" / "figure4_resolved.yaml",
    device=DEVICE,
    repo_root=ROOT,
)
time_slice_dir = render_hspc_time_slices(
    config_path=local_cfg,
    output_dir=run_dir / "figures" / "time_slices",
    repo_root=ROOT,
)
time_slice_dir


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "time_slices" / "time_slice_panels.png")


## Perturbation setup

The fitted dynamics and cross-modality map are evaluated using the same source cells and UMAP embeddings across conditions.


In [ ]:
import shutil
from CytoBridge.pl.tf.context import TFRunContext

ctx = TFRunContext.from_run_dir(run_dir)
trajectory_index_csv = ROOT / "configs" / "analysis_inputs" / "hspc_mkp_paper8_indices.csv"
trajectory_dir = run_dir / "figures" / "tf" / "paper_mkp_trajectory"
trajectory_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    ROOT / "paper_data" / "umap" / DATASET / "perturb_primary.pkl",
    trajectory_dir / "umap_model_primary.pkl",
)
shutil.copy2(
    ROOT / "paper_data" / "umap" / DATASET / "perturb_secondary.pkl",
    trajectory_dir / "umap_model_secondary.pkl",
)


## MkP program perturbation

Negative and positive perturbation scores reduce and increase the selected MkP program, respectively. The unperturbed prediction provides the reference.


In [ ]:
from CytoBridge.pl.tf.perturb import plot_perturbation_trajectories

with suppress_live_plot_output():
    perturb_trajectories = plot_perturbation_trajectories(
        ctx,
        indices_csv=str(trajectory_index_csv),
        scores=[-40, 0, 40],
        sample_n=8,
        seed=42,
        device=DEVICE,
        output_dir=str(trajectory_dir),
        dim_reduction="umap",
        modalities=("primary",),
    )
perturb_trajectories


### Reduced MkP program (z = -40)


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "tf" / "paper_mkp_trajectory" / "perturb_index_traj_primary__g-mk-union__z-m40__dr-umap.png")


### Unperturbed reference (z = 0)


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "tf" / "paper_mkp_trajectory" / "perturb_index_traj_primary__g-mk-union__z-0__dr-umap.png")


### Increased MkP program (z = 40)


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "tf" / "paper_mkp_trajectory" / "perturb_index_traj_primary__g-mk-union__z-40__dr-umap.png")


## Marker-level perturbation analysis

The analysis links perturbations of MkP-associated genes to predicted responses in the protein modality.


In [ ]:
import yaml
from CytoBridge.pl.tf.perturb_pipeline import run_perturbation_pipeline_for_run

with (ROOT / "configs" / "perturbation" / "hspc_31800_primary.yaml").open(encoding="utf-8") as stream:
    perturb_cfg = yaml.safe_load(stream)["perturb_pipeline"]

fate_index_csv = ROOT / "configs" / "analysis_inputs" / "hspc_final_mkp_indices.csv"
perturb_cfg.update(
    indices_csv=str(fate_index_csv),
    use_indices_csv=True,
    device=DEVICE,
    target_group="Mk_union",
    output_table_dir=str(run_dir / "tables" / "tf" / "paper_primary"),
    output_figure_dir=str(run_dir / "figures" / "tf" / "paper_primary"),
)
if "fate_sweep" in perturb_cfg:
    perturb_cfg["fate_sweep"]["enabled"] = False

with suppress_live_plot_output():
    perturb = run_perturbation_pipeline_for_run(str(run_dir), cfg=perturb_cfg)
perturb["manifest_json"]


## Predicted fate composition

The heatmap shows changes in terminal cell-type composition relative to the unperturbed prediction across nine perturbation scores.


In [ ]:
from CytoBridge.pl.tf.perturb import build_perturbation_fate_sweep

with suppress_live_plot_output():
    fate = build_perturbation_fate_sweep(
        ctx,
        indices_csv=str(ROOT / "configs" / "analysis_inputs" / "hspc_final_mkp_indices.csv"),
        target_group="Mk_union",
        target_genes=["ITGA2B", "GP1BA", "PF4", "PPBP"],
        scores=[-40, -30, -20, -10, 0, 10, 20, 30, 40],
        sample_n="all",
        seed=42,
        device=DEVICE,
        output_table_dir=str(run_dir / "tables" / "tf" / "paper_mkp_fate"),
        output_figure_dir=str(run_dir / "figures" / "tf" / "paper_mkp_fate"),
        sweep_cell_types=["EryP", "HSC", "MasP", "MkP", "MoP", "NeuP"],
        focus_curve_cell_types=["EryP", "HSC", "MasP", "MkP", "MoP", "NeuP"],
        focus_curve_scores=[-40, 0, 40],
        heatmap_z_order=[-40, -30, -20, -10, 0, 10, 20, 30, 40],
        heatmap_orientation="z_by_cell_type",
        heatmap_cell_type_order=["HSC", "MoP", "MasP", "NeuP", "MkP", "EryP"],
        heatmap_vmax_percent=40,
        prob_heatmap_value_col="delta_rate",
    )
fate


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(run_dir / "figures" / "tf" / "paper_mkp_fate" / "perturbation_sweep_heatmap_prob.png")


## ITGA2B–CD41 response

The curves compare predicted ITGA2B and CD41 dynamics across perturbation scores.


In [ ]:
from CytoBridge.pl.tf.perturb_pipeline import plot_saved_perturbation_batch_curves

endpoint_table_dir = (
    run_dir / "tables" / "tf" / "paper_primary" / "primary" / "batch" / "a_itga2b_cd41"
)
with suppress_live_plot_output():
    itga2b_curve = plot_saved_perturbation_batch_curves(
        endpoint_table_dir,
        run_dir / "figures" / "tf" / "paper_primary" / "primary" / "batch" /
        "perturb_batch_curve__task-a-itga2b-cd41__allz.png",
        scores=[-40, -20, 0, 20, 40],
        primary_var="ITGA2B",
        secondary_var="CD41",
        title_suffix="ITGA2B → CD41",
    )
itga2b_curve


In [ ]:
from mofi_notebook import show_paper_panel
show_paper_panel(itga2b_curve["output"])


## Cross-modality transformation

Local directions in RNA space are mapped to protein space and vice versa to identify related molecular changes.


In [ ]:
from CytoBridge.pl.tf.regulation_umap import build_regulation_umap_outputs
from CytoBridge.manuscript import build_transformation_summary

with suppress_live_plot_output():
    transformation = build_regulation_umap_outputs(
        ctx=ctx,
        out_fig_dir=run_dir / "figures" / "tf" / "transformation",
        cfg_obj={"device": DEVICE, "regulation_umap": {
            "enabled": True,
            "obs_key": "cell_type",
            "obs_value": "MasP",
            "celltype_time_scope": "all_times",
            "seed": 42,
            "device": DEVICE,
            "max_selected_cells": 256,
            "mapper_batch_size": 128,
            "include_hessian_term": False,
            "umap_neighbors": 10,
            "umap_min_dist": 0.1,
            "umap_seed": 42,
        }},
    )

transformation_csv = run_dir / "tables" / "tf" / "transformation" / "transformation_summary.csv"
transformation_summary = build_transformation_summary(
    transformation,
    selected_csv=ROOT / "configs" / "analysis_inputs" / "hspc_transformation_genes.csv",
    output_csv=transformation_csv,
    repo_root=ROOT,
)
transformation_summary
